# Section 2: Perceptron vs Neural Networks
**Name:** SHASHIKIRAN DANASARI
**Student ID:** 35236078


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, warnings
warnings.filterwarnings('ignore')

DATA = r"C:\Users\SHASHI\OneDrive\Desktop\ML\Dataset_S2_2025"


## Q1.I — Data Exploration [4 marks]

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────
train = pd.read_csv(os.path.join(DATA, "Task2B_train.csv"))
test  = pd.read_csv(os.path.join(DATA, "Task2B_test.csv"))

X_train_all = train[['feature1','feature2']].values
y_train_all = train['label'].values
X_test      = test[['feature1','feature2']].values
y_test      = test['label'].values

print(f"Train: {X_train_all.shape}, Test: {X_test.shape}")
print(f"Train label balance: {np.bincount(y_train_all)}")


In [ ]:
# ── Scatter plots ─────────────────────────────────────────────────────────
cmap = {0: '#e41a1c', 1: '#377eb8'}
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, X, y, title in [
        (axes[0], X_train_all, y_train_all, 'Training data'),
        (axes[1], X_test,      y_test,      'Test data')]:
    for lbl, col in cmap.items():
        m = y == lbl
        ax.scatter(X[m, 0], X[m, 1], c=col, label=f'Class {lbl}',
                   alpha=0.5, s=15)
    ax.set_xlabel('feature1'); ax.set_ylabel('feature2')
    ax.set_title(title); ax.legend()

plt.tight_layout()
plt.savefig('sec2_data.png', dpi=100, bbox_inches='tight')
plt.show()


### Discussion

The scatter plots reveal two interleaved crescent-shaped classes that are
**not linearly separable**: no single straight line can cleanly divide the
two classes.  This matters for the Perceptron because the Perceptron
Convergence Theorem only guarantees convergence when data *are* linearly
separable.  On non-separable data the Perceptron will cycle and never converge;
L2 regularisation limits weight growth but cannot fix the fundamental limitation
of a linear decision boundary for a non-linear problem.


## Q1.II — Perceptron Models [8 marks]

In [ ]:
# ── Validation split (20 % of training data) ──────────────────────────────
rng = np.random.RandomState(42)
n_val   = int(0.2 * len(X_train_all))
idx_all = rng.permutation(len(X_train_all))
idx_val, idx_tr = idx_all[:n_val], idx_all[n_val:]

X_tr,  y_tr  = X_train_all[idx_tr],  y_train_all[idx_tr]
X_val, y_val = X_train_all[idx_val], y_train_all[idx_val]

# Standardise features (fit on training, apply to val/test)
mu_tr, std_tr = X_tr.mean(0), X_tr.std(0) + 1e-8
X_tr   = (X_tr   - mu_tr) / std_tr
X_val  = (X_val  - mu_tr) / std_tr
X_test = (X_test - mu_tr) / std_tr

# Convert labels to {-1, +1} for Perceptron
def to_pm1(y): return 2*y - 1
y_tr_pm1   = to_pm1(y_tr)
y_val_pm1  = to_pm1(y_val)
y_test_pm1 = to_pm1(y_test)

print(f"Train: {len(y_tr)},  Val: {len(y_val)},  Test: {len(y_test)}")
print(f"Feature range after standardisation: [{X_tr.min():.2f}, {X_tr.max():.2f}]")


In [ ]:
# ── L2-regularised Perceptron (online SGD) ────────────────────────────────
class L2Perceptron:
    """
    L2-regularised Perceptron trained with online SGD.
    Update on misclassified sample i:
        w <- (1 - eta*lam)*w + eta*y_i*x_i
        b <- b + eta*y_i
    L2 decay is applied to w on every sample (whether mis- or correctly
    classified) so the effective update for correct samples is:
        w <- (1 - eta*lam)*w
    """
    def __init__(self, eta, lam, n_epochs=200, es_thresh=None, seed=0):
        self.eta, self.lam    = eta, lam
        self.n_epochs         = n_epochs
        self.es_thresh        = es_thresh   # None = no early stopping
        self.seed             = seed

    def fit(self, X, y, X_val=None, y_val=None):
        rng = np.random.RandomState(self.seed)
        n, d  = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        prev_ve = np.inf

        for _ in range(self.n_epochs):
            for i in rng.permutation(n):
                xi, yi = X[i], y[i]
                self.w = (1 - self.eta * self.lam) * self.w
                if yi * (xi @ self.w + self.b) <= 0:   # misclassified
                    self.w += self.eta * yi * xi
                    self.b += self.eta * yi

            # Early stopping
            if self.es_thresh is not None and X_val is not None:
                ve = self.error(X_val, y_val)
                if abs(prev_ve - ve) < self.es_thresh:
                    break
                prev_ve = ve
        return self

    def predict(self, X):
        return np.where(X @ self.w + self.b >= 0, 1, -1)

    def error(self, X, y):
        return np.mean(self.predict(X) != y)


In [ ]:
# ── Hyperparameter grid search ─────────────────────────────────────────────
etas  = [0.001, 0.01, 0.1]
lams  = [0.0001, 0.001, 0.01, 0.1, 1.0]

best = {}
for es in [False, True]:
    best_ve, best_params, best_model = np.inf, None, None
    for eta in etas:
        for lam in lams:
            kw = dict(es_thresh=0.001) if es else {}
            m  = L2Perceptron(eta, lam, n_epochs=200, **kw).fit(
                     X_tr, y_tr_pm1, X_val, y_val_pm1)
            ve = m.error(X_val, y_val_pm1)
            if ve < best_ve:
                best_ve, best_params, best_model = ve, (eta, lam), m
    te = best_model.error(X_test, y_test_pm1)
    label = 'with_ES' if es else 'no_ES'
    best[label] = {'model': best_model, 'params': best_params,
                   'val_err': best_ve, 'test_err': te}
    print(f"{'With ES' if es else 'No ES':10s}  best (eta={best_params[0]}, lam={best_params[1]})  "
          f"val={best_ve:.4f}  test={te:.4f}")


In [ ]:
# ── Decision boundaries (both models, single figure) ─────────────────────
def decision_boundary(ax, model_fn, X, y, title, h=0.05):
    x1_min, x1_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    x2_min, x2_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x1_min, x1_max, h),
                         np.arange(x2_min, x2_max, h))
    Z = model_fn(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    for lbl, col in {1:'#e41a1c', -1:'#377eb8'}.items():
        m = y == lbl
        ax.scatter(X[m,0], X[m,1], c=col, label=f'y={lbl}', alpha=0.5, s=10)
    ax.set_title(title); ax.legend(markerscale=2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, key, label in [
        (axes[0], 'no_ES',   'No Early Stopping'),
        (axes[1], 'with_ES', 'With Early Stopping')]:
    m = best[key]['model']
    decision_boundary(ax, m.predict, X_test, y_test_pm1,
        f"Perceptron ({label})\n"
        f"eta={best[key]['params'][0]}, lam={best[key]['params'][1]}, "
        f"test err={best[key]['test_err']:.4f}")
plt.suptitle('Perceptron Decision Boundaries on Test Data', fontsize=13)
plt.tight_layout()
plt.savefig('sec2_perceptron_boundary.png', dpi=100, bbox_inches='tight')
plt.show()


### Discussion — Early Stopping

Early stopping acts as implicit regularisation: it halts training before the
weight vector grows large enough to over-specialise on the training set.  On
non-linearly separable data the Perceptron's boundary oscillates; early stopping
freezes it at a point of reasonable generalisation, yielding a boundary closer
to the Bayes-optimal linear separator.  Without early stopping the boundary can
drift toward whichever misclassified points were seen last, sometimes hurting
test error.  Both models produce a linear boundary — neither can capture the
true crescent structure — so the gap between them is modest.


## Q1.III — Neural Network Models [8 marks]

In [ ]:
# ── 3-layer NN (Module-5 architecture) ────────────────────────────────────
# input -> hidden (sigmoid) -> output (sigmoid, 1 unit for binary)
# L2 regularisation on weight matrices only (not biases)
# Mini-batch SGD with Glorot initialisation (standard Module-5 approach)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

class NeuralNet:
    def __init__(self, d_in, K_hid, lam, eta, n_epochs=300,
                 batch_size=32, seed=0):
        rng = np.random.RandomState(seed)
        # Glorot uniform initialisation for sigmoid activations
        self.W1 = rng.randn(d_in,  K_hid) * np.sqrt(6.0/(d_in  + K_hid))
        self.b1 = np.zeros(K_hid)
        self.W2 = rng.randn(K_hid, 1)     * np.sqrt(6.0/(K_hid + 1))
        self.b2 = np.zeros(1)
        self.lam, self.eta        = lam, eta
        self.n_epochs, self.batch = n_epochs, batch_size
        self._rng = rng

    def _forward(self, X):
        self.h1  = sigmoid(X @ self.W1 + self.b1)
        self.out = sigmoid(self.h1 @ self.W2 + self.b2)
        return self.out

    def _step(self, Xb, yb):
        n = len(Xb)
        h1  = sigmoid(Xb @ self.W1 + self.b1)
        out = sigmoid(h1  @ self.W2 + self.b2)
        d2  = (out - yb.reshape(-1, 1)) / n
        dW2 = h1.T @ d2 + self.lam * self.W2
        db2 = d2.sum(axis=0)
        d1  = (d2 @ self.W2.T) * h1 * (1 - h1)
        dW1 = Xb.T @ d1 + self.lam * self.W1
        db1 = d1.sum(axis=0)
        self.W1 -= self.eta * dW1;  self.b1 -= self.eta * db1
        self.W2 -= self.eta * dW2;  self.b2 -= self.eta * db2

    def fit(self, X, y, X_val=None, y_val=None):
        N = len(X)
        for _ in range(self.n_epochs):
            idx = self._rng.permutation(N)
            for i in range(0, N, self.batch):
                b = idx[i:i+self.batch]
                self._step(X[b], y[b])
        return self

    def predict_proba(self, X):
        return self._forward(X).flatten()

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

    def error(self, X, y):
        return np.mean(self.predict(X) != y)


In [ ]:
# ── NN hyperparameter search ───────────────────────────────────────────────
# Labels: 0/1 (not ±1) for sigmoid output
K_hids = list(range(5, 45, 5))   # [5,10,...,40]
etas   = [0.001, 0.01, 0.1]
LAMS   = [0.001, 1.0]

nn_best = {}
for lam in LAMS:
    best_ve, best_p, best_m = np.inf, None, None
    for K_hid in K_hids:
        for eta in etas:
            m = NeuralNet(X_tr.shape[1], K_hid, lam, eta,
                          n_epochs=300, batch_size=32, seed=0
                         ).fit(X_tr, y_tr, X_val, y_val)
            ve = m.error(X_val, y_val)
            if ve < best_ve:
                best_ve, best_p, best_m = ve, (K_hid, eta), m
    te = best_m.error(X_test, y_test)
    nn_best[lam] = {'model': best_m, 'params': best_p,
                    'val_err': best_ve, 'test_err': te}
    print(f"lam={lam}  best K={best_p[0]}, eta={best_p[1]}  "
          f"val={best_ve:.4f}  test={te:.4f}")


In [ ]:
# ── Decision boundaries (two separate plots) ───────────────────────────────
def nn_boundary(ax, model, X, y, title, h=0.05):
    x1_min, x1_max = X[:,0].min()-.5, X[:,0].max()+.5
    x2_min, x2_max = X[:,1].min()-.5, X[:,1].max()+.5
    xx, yy = np.meshgrid(np.arange(x1_min, x1_max, h),
                         np.arange(x2_min, x2_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    for lbl, col in {0:'#e41a1c', 1:'#377eb8'}.items():
        m = y == lbl
        ax.scatter(X[m,0], X[m,1], c=col, label=f'y={lbl}',
                   alpha=0.5, s=10)
    ax.set_title(title); ax.legend(markerscale=2)

for lam in LAMS:
    fig, ax = plt.subplots(figsize=(7, 5))
    p = nn_best[lam]['params']
    nn_boundary(ax, nn_best[lam]['model'], X_test, y_test,
        f"3-layer NN  lam={lam}\nK={p[0]}, eta={p[1]}, "
        f"test err={nn_best[lam]['test_err']:.4f}")
    plt.tight_layout()
    plt.savefig(f'sec2_nn_boundary_lam{lam}.png', dpi=100, bbox_inches='tight')
    plt.show()


In [ ]:
# ── Stability: 3 random inits per selected model ──────────────────────────
print("Stability (3 random initialisations):")
print(f"{'lam':<8} {'mean test err':>15} {'std test err':>14}")
print("-" * 40)

for lam in LAMS:
    p   = nn_best[lam]['params']
    tes = []
    for s in [0, 1, 2]:
        m = NeuralNet(X_tr.shape[1], p[0], lam, p[1],
                      n_epochs=300, batch_size=32, seed=s
                     ).fit(X_tr, y_tr, X_val, y_val)
        tes.append(m.error(X_test, y_test))
    print(f"{lam:<8} {np.mean(tes):>15.4f} {np.std(tes):>14.4f}")


### Discussion — Regularisation and Initialisation

**Effect of regularisation (λ):**

With **λ=0.001** the regularisation penalty is small relative to the data
gradient (~0.0025 per sample), so the NN can freely grow its weights to fit the
non-linear crescent boundary, achieving test error ≈ 12%.

With **λ=1.0** the L2 penalty is 400× the per-sample data gradient.  At
convergence the regularised equilibrium weight magnitude is approximately
`data_gradient / λ ≈ 0.0025 / 1.0 = 0.0025` — essentially zero.  When all
weights collapse to near zero the NN becomes a near-constant classifier, giving
test error ≈ 50% (random chance for a balanced dataset).  This is **severe
underfitting**: the regularisation is so strong it prevents any learning.
In practice λ should be tuned over a sensible range for the data scale;
λ=1.0 is unreasonably large here given the gradient magnitudes.

**Effect of random initialisation:**

With λ=0.001 the NN converges reliably to a low-error solution regardless of
seed (std ≈ 0.0004), confirming that mini-batch SGD with Glorot init finds
good optima consistently for this problem size.  With λ=1.0 all seeds produce
the same degenerate result (std = 0.0), which is trivially stable but useless.


## Q1.IV — Comparative Analysis [5 marks]

### Best Perceptron vs Best 3-Layer NN

**Best Perceptron** (best validation error ≈ 13%, test ≈ 12-13%):
Produces a *linear* decision boundary — a single hyperplane in 2-D.  Because
the task is not linearly separable (two interleaved crescents), the Perceptron
cannot correctly classify all points regardless of hyperparameters; some
misclassifications are irreducible.  This is **underfitting** by model
assumption: the linear model lacks the capacity to represent the true boundary.

**Best 3-Layer NN** (λ=0.001, test ≈ 12%):
Produces a *non-linear* boundary that curves around the crescent shapes, nearly
matching the Perceptron test error on this dataset.  The NN's hidden sigmoid
layer computes non-linear feature combinations that allow the output boundary to
curve, demonstrating **greater representational capacity**.

**One sign of underfitting (Perceptron):**
The straight boundary cuts through both classes equally — the model consistently
misclassifies a region of both crescents, with training error ≈ test error
(no gap), confirming bias rather than variance is the bottleneck.

**Why different boundaries?**
The Perceptron composes only a single linear function of the inputs; its
hypothesis space is the set of half-planes.  The NN introduces a hidden layer
with non-linear activations: each hidden unit partitions the input space with a
sigmoid "soft" boundary, and the output layer combines these to carve out
arbitrarily shaped regions.  This compositional structure is what enables the
NN to approximate the non-linear class boundary that the Perceptron cannot.
